# MedicalQA — Claude Opus 5 LLM Judge

This notebook runs the current caregiver evaluation rubric using `claude-opus-5`. Standalone is binary-only and is judged in a separate call that never receives the transcript. All other metrics receive the source transcript.

The run checkpoints after every completed Q&A to Google Drive. Re-running the paid cell resumes instead of paying for completed pairs again.

Before starting, add `ANTHROPIC_API_KEY` in Colab's **Secrets** panel (key icon on the left) and enable notebook access. Never paste the key into this notebook.

In [ ]:
%pip -q install -U anthropic

## Upload the prepared data bundle

Choose `Claude_Opus_Judge_Colab_Bundle.zip` when the upload box opens. It contains only the judge code, Q&A data and 25 transcripts—not the complete repository.

In [ ]:
from google.colab import files
from pathlib import Path
import zipfile

uploaded = files.upload()
bundle_name = next((name for name in uploaded if name.endswith('.zip')), None)
assert bundle_name, 'Please upload Claude_Opus_Judge_Colab_Bundle.zip'
with zipfile.ZipFile('/content/' + bundle_name) as archive:
    archive.extractall('/content')
REPO = Path('/content/MedicalQA')
assert (REPO / 'eval/llm_judge/claude_judge.py').exists(), REPO
print('MedicalQA judge bundle ready:', REPO)

## Connect your key and persistent output folder

In [ ]:
from google.colab import drive, userdata
import os

drive.mount('/content/drive')
api_key = userdata.get('ANTHROPIC_API_KEY')
assert api_key, 'Add ANTHROPIC_API_KEY in the Colab Secrets panel first.'
os.environ['ANTHROPIC_API_KEY'] = api_key
OUTPUT_DIR = Path('/content/drive/MyDrive/MedicalQA_Claude_Judge')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print('Checkpoints and reports will be saved in:', OUTPUT_DIR)

## Choose the evaluation scope

Keep `SCOPE = 'v3'` to judge all 226 v3 pairs. Change it to `'v1-v2-v3'` only when you want the full 677-record prompt-ablation report comparable to the earlier DeepSeek artifact.

Set `FIRST_N = 40` for a small latest-v3 calibration run; use `0` for the complete corpus.

**Corpus warning:** the checked-in Netlify human-evaluation package currently contains the 225-pair v2 snapshot, not these latest 226 v3 pairs. Do not compare its Supabase rows with this Claude v3 run by position. The report accepts human data only for Q&As whose ID or exact question-and-answer text matches.

In [ ]:
SCOPE = 'v3'              # 'v3' or 'v1-v2-v3'
FIRST_N = 0               # 0 = all; 40 = small latest-v3 calibration
WORKERS = 2               # reduce to 1 if your API tier rate-limits
EFFORT = 'high'
MODEL = 'claude-opus-5'

VERSIONS = ['v3'] if SCOPE == 'v3' else ['v1', 'v2', 'v3']
suffix = 'full' if FIRST_N == 0 else f'first{FIRST_N}'
RUN_NAME = f"claude_opus5_{'_'.join(VERSIONS)}_{suffix}"
print('Versions:', VERSIONS, '| run:', RUN_NAME)

## Validate the corpus — free, no API call

In [ ]:
import subprocess, sys

runner = REPO / 'eval/llm_judge/claude_judge.py'
base_command = [
    sys.executable, str(runner),
    '--versions', *VERSIONS,
    '--model', MODEL, '--effort', EFFORT,
    '--workers', str(WORKERS),
    '--first-n', str(FIRST_N),
    '--output-dir', str(OUTPUT_DIR),
    '--run-name', RUN_NAME,
]
subprocess.run(base_command + ['--dry-run'], check=True)

## Optional three-pair paid smoke test

Run this once before the full job. It verifies account billing, API access, structured output and Drive checkpointing.

In [ ]:
smoke_name = RUN_NAME + '_smoke3'
smoke_command = [arg if arg != RUN_NAME else smoke_name for arg in base_command]
subprocess.run(smoke_command + ['--limit', '3'], check=True)

## Run the paid evaluation

Running this cell starts the complete API job. If Colab disconnects or the API rate-limits, reconnect and run the setup/configuration cells followed by this cell again. The fixed run name automatically resumes the JSONL checkpoint in Drive.

In [ ]:
subprocess.run(base_command, check=True)

## Build the artifact-style report

The report is deterministic: it summarizes the saved Claude ratings without asking another model to rewrite or reinterpret the numbers. Set the optional CSV paths if you later want Claude-versus-DeepSeek or Claude-versus-human agreement tables. A human file may be either the evaluation site's downloaded CSV or a Supabase `ratings_v3_final` CSV export.

In [ ]:
CLAUDE_CSV = OUTPUT_DIR / f'{RUN_NAME}.csv'
DEEPSEEK_CSV = ''          # optional Drive path to the earlier DeepSeek CSV
HUMAN_CSV = ''             # optional Drive path to human/Supabase CSV
REPORT_HTML = OUTPUT_DIR / f'{RUN_NAME}_report.html'
TITLE = ('Claude Opus 5 — SingleAgent v3 Full-Corpus Evaluation'
         if SCOPE == 'v3' else 'Standalone Criterion Ablation — Claude Opus 5')

report_command = [
    sys.executable, str(REPO / 'eval/llm_judge/build_claude_report.py'),
    '--claude-csv', str(CLAUDE_CSV),
    '--output', str(REPORT_HTML), '--title', TITLE,
]
if DEEPSEEK_CSV:
    report_command += ['--deepseek-csv', DEEPSEEK_CSV]
if HUMAN_CSV:
    report_command += ['--human-csv', HUMAN_CSV]
subprocess.run(report_command, check=True)

## Preview and download

In [ ]:
from IPython.display import HTML, display
display(HTML(REPORT_HTML.read_text(encoding='utf-8')))
print('Saved in Drive:', REPORT_HTML)
# Uncomment to download a local copy:
# files.download(str(REPORT_HTML))